## 1. Configuración del archivo de logs
Todos los eventos y errores se guardan en `software_fj.log`. Es el **único** archivo que usa el sistema
(la información de clientes, servicios y reservas vive en objetos y listas).

In [1]:
# --- Importaciones ---------------------------------------------------------
import logging                                   # Módulo estándar para registrar eventos en archivos
import re                                        # Expresiones regulares para validar correo y teléfono
from abc import ABC, abstractmethod              # Herramientas para crear clases abstractas
from datetime import datetime                    # Fecha y hora de creación de entidades
from functools import singledispatchmethod      # Permite "sobrecargar" métodos según el tipo del argumento

ARCHIVO_LOG = "software_fj.log"                  # Nombre del archivo donde se registran eventos y errores

logger = logging.getLogger("SoftwareFJ")         # Se crea (u obtiene) un logger con nombre propio
logger.setLevel(logging.DEBUG)                   # Se registran todos los niveles: DEBUG, INFO, WARNING, ERROR
logger.handlers.clear()                          # Evita handlers duplicados si la celda se ejecuta varias veces

manejador = logging.FileHandler(ARCHIVO_LOG, mode="w", encoding="utf-8")  # "w": log limpio en cada ejecución
formato = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s",  # Formato: fecha | nivel | mensaje
                            datefmt="%Y-%m-%d %H:%M:%S")
manejador.setFormatter(formato)                  # Se asigna el formato al manejador de archivo
logger.addHandler(manejador)                     # Se conecta el manejador al logger

logger.info("===== Inicio del Sistema Software FJ =====")  # Primer evento registrado
print(f"Logger configurado. Los eventos se guardan en '{ARCHIVO_LOG}'.")

Logger configurado. Los eventos se guardan en 'software_fj.log'.


## 2. Excepciones personalizadas
Se crea una jerarquía propia: todas heredan de `SoftwareFJError`, así el sistema puede capturar
cualquier error del dominio con un solo `except` sin ocultar errores inesperados de Python.

In [ ]:
class SoftwareFJError(Exception):
    """Excepción base del sistema. Todas las demás heredan de ella."""
    def __init__(self, mensaje, codigo="FJ-000"):
        super().__init__(mensaje)                # Guarda el mensaje en la excepción estándar
        self.codigo = codigo                     # Código interno para identificar el tipo de error

    def __str__(self):
        return f"[{self.codigo}] {self.args[0]}" # Muestra el código junto al mensaje


class ClienteInvalidoError(SoftwareFJError):
    """Datos personales del cliente inválidos o faltantes."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-100")


class ServicioInvalidoError(SoftwareFJError):
    """Servicio mal configurado (tarifa, capacidad, parámetros)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-200")


class ServicioNoDisponibleError(SoftwareFJError):
    """El servicio existe pero no está disponible para reservar."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-210")


class ReservaInvalidaError(SoftwareFJError):
    """Intento de reserva con datos incorrectos."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-300")


class OperacionNoPermitidaError(SoftwareFJError):
    """Operación prohibida según el estado actual (ej.: cancelar algo ya procesado)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-400")


class CalculoInconsistenteError(SoftwareFJError):
    """Un cálculo de costo dio un resultado ilógico (negativo, descuento > 100 %, etc.)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-500")


print("Excepciones personalizadas definidas:",
      [c.__name__ for c in SoftwareFJError.__subclasses__()])

Excepciones personalizadas definidas: ['ClienteInvalidoError', 'ServicioInvalidoError', 'ServicioNoDisponibleError', 'ReservaInvalidaError', 'OperacionNoPermitidaError', 'CalculoInconsistenteError']
